# Task 1 Kaggle Training Notebook (Unsloth)

Notebook nay train **Gemma 4 12B** cho bai toan **Task 1** tren **AvaMERG + ESConv** theo workflow **Unsloth-first**.

Vi sao dung Unsloth o day:
- giam VRAM va train nhanh hon cho LoRA/QLoRA
- hop hon voi Kaggle hon so voi viec tu rap bitsandbytes + transformers
- van giu duoc dataset/prompt/research framing cua repo

Thu tu nen chay:
1. Cell 1 cai moi truong Unsloth
2. Restart kernel
3. Cell 3-5 kiem tra version + login + test model load
4. Cell 6-8 tai va inspect du lieu
5. Cell 9 dump prompt examples
6. Cell 10 build HF Dataset
7. Cell 11 smoke train
8. Cell 12 train that


In [ ]:
import os
REPO_URL = "https://github.com/QuangVoAI/multimodal-empathy-mental-health.git"
REPO_DIR = "/kaggle/working/multimodal-empathy-mental-health"
!rm -rf {REPO_DIR}
!git clone {REPO_URL} {REPO_DIR}
%cd /kaggle/working/multimodal-empathy-mental-health
%pip uninstall -y datasets transformers huggingface_hub accelerate peft bitsandbytes sentencepiece tokenizers torchvision unsloth unsloth_zoo trl Pillow
%pip install --no-cache-dir --force-reinstall -r requirements_kaggle_unsloth.txt
%pip install --no-cache-dir --force-reinstall --no-deps "transformers>=5.10.2"


## Important: restart the Kaggle kernel now

Sau khi cai package xong, hay restart kernel roi moi chay tiep.


In [ ]:
%cd /kaggle/working/multimodal-empathy-mental-health
import unsloth
import torch, transformers, datasets, trl
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("unsloth:", getattr(unsloth, "__version__", "unknown"))
print("datasets:", datasets.__version__)
print("trl:", trl.__version__)


In [ ]:
from huggingface_hub import login
HF_TOKEN = "YOUR_HF_TOKEN"
login(HF_TOKEN)


In [ ]:
from unsloth import FastLanguageModel

MODEL_ID = "unsloth/gemma-4-12b-it"
MAX_SEQ_LENGTH = 1536
LOAD_IN_4BIT = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = LOAD_IN_4BIT,
    token = HF_TOKEN,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Loaded:", MODEL_ID)
print("Tokenizer:", type(tokenizer).__name__)
print("Pad token:", tokenizer.pad_token)


In [ ]:
!bash scripts/download_avamerg.sh
!bash scripts/download_esconv.sh
!mkdir -p outputs/sft outputs/eval


In [ ]:
import json
from pathlib import Path

ava = json.loads(Path("data/raw/avamerg/train.json").read_text(encoding="utf-8"))
esc = json.loads(Path("data/raw/esconv/ESConv.json").read_text(encoding="utf-8"))
print("AvaMERG samples:", len(ava))
print("AvaMERG first keys:", list(ava[0].keys()))
print("ESConv dialogues:", len(esc))
print("ESConv first keys:", list(esc[0].keys()))


In [ ]:
from src.data import AvaMERGDataset, ESConvDataset
from src.models.gemma_merg import DEFAULT_SYSTEM_PROMPT, GemmaMERG
from pathlib import Path
import json

prompt_builder = GemmaMERG.__new__(GemmaMERG)
prompt_builder.model_name_or_path = MODEL_ID
prompt_builder.tokenizer = tokenizer

ava_ds = AvaMERGDataset(root="data/raw/avamerg", split="train", use_multimodal=False)
esconv_ds = ESConvDataset("data/raw/esconv/ESConv.json")

examples = []
for ds in [ava_ds, esconv_ds]:
    for idx in range(min(2, len(ds))):
        sample = ds[idx]
        user_prompt = prompt_builder.build_user_prompt(sample)
        messages = [
            {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": sample["response"]},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        examples.append({"sample_id": sample.get("sample_id"), "source_dataset": sample.get("source_dataset"), "text": text})
Path("outputs/sft/debug_joint").mkdir(parents=True, exist_ok=True)
Path("outputs/sft/debug_joint/example_prompts_unsloth.json").write_text(json.dumps(examples, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved examples to outputs/sft/debug_joint/example_prompts_unsloth.json")


In [ ]:
from datasets import Dataset

def to_chat_text(sample):
    user_prompt = prompt_builder.build_user_prompt(sample)
    messages = [
        {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": sample["response"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

rows = []
for ds in [ava_ds, esconv_ds]:
    for idx in range(len(ds)):
        sample = ds[idx]
        rows.append({"text": to_chat_text(sample), "source_dataset": sample.get("source_dataset")})
train_ds = Dataset.from_list(rows)
print(train_ds)
print(train_ds[0]["text"][:800])


In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

smoke_ds = train_ds.select(range(min(32, len(train_ds))))
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = smoke_ds,
    dataset_text_field = "text",
    args = SFTConfig(
        output_dir = "outputs/sft/unsloth_smoke",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 2,
        learning_rate = 1e-4,
        max_steps = 1,
        logging_steps = 1,
        save_strategy = "no",
        bf16 = torch.cuda.is_available(),
        report_to = [],
    ),
)
trainer.train()


In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

# Run this after the smoke train succeeds.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = LOAD_IN_4BIT,
    token = HF_TOKEN,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    dataset_text_field = "text",
    args = SFTConfig(
        output_dir = "outputs/sft/task1_unsloth_run",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        learning_rate = 1e-4,
        num_train_epochs = 1,
        logging_steps = 10,
        save_steps = 100,
        bf16 = torch.cuda.is_available(),
        report_to = [],
    ),
)
trainer.train()
trainer.model.save_pretrained("outputs/sft/task1_unsloth_run/final")
tokenizer.save_pretrained("outputs/sft/task1_unsloth_run/final")
